# Tensor API 矩阵算子优化实践

## 概述

本课程介绍如何使用 Tensor API 开发矩阵乘法算子，并用不同优化手段逐步提升算子性能。

矩阵乘法算子实现 `C = A * B`，输入矩阵 `A`、`B` 和输出矩阵 `C` 的尺寸均为 `[8192, 8192]`，数据类型均为 `half`。课程按以下路线展开：

1. 先分析矩阵乘法算子的接口、数据类型和数据排布约定；
2. 介绍 Tensor API 中 Tensor、Layout、Slice 和 Copy、Mmad 的基本使用方式；
3. 完成单核版本，作为后续优化的基线版本；
4. 在单核版本基础上通过多核切分提高 Cube 核并行度；
5. 在多核版本基础上开启 double buffer，隐藏数据搬运时间，提高 Cube 核利用率。

### 前置要求

- 已学习本课程第 3 章 SIMD 编程和 Tensor 编程，了解 Ascend C 的核函数开发和基本工程结构。
- 了解 Cube 核矩阵计算的基本概念：GM、L1、L0A/L0B、L0C 各级存储的层次关系。
- 已配置 CANN 开发环境，并能访问 Ascend 950PR/DT 设备。

### 学习目标

完成本小节后，开发者应能够：

1. 理解 Tensor API 中 `Tensor`、`Layout`、`Slice` 对矩阵块的描述和切块方式；
2. 掌握 `GM → L1 → L0A/L0B → Mmad → L0C → GM` 的基本数据通路；
3. 掌握「单核 → 多核 → double buffer」的优化原理，能独立实现并对比性能差异；
4. 能完成 stepK 大包搬运优化，减少 GM→L1 搬运指令数。

### 本节内容

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">章节</th>
      <th align="left">内容</th>
      <th align="left">学习期望</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><a href="#1-算子接口分析">1. 算子接口分析</a></td>
      <td>输入输出约定、数据类型和精度校验策略</td>
      <td>明确算子的数据规格和校验方式</td>
    </tr>
    <tr>
      <td><a href="#2-tensor-api-矩阵乘法样例工程">2. 样例工程</a></td>
      <td>工程目录、CMake、Host/Device 代码和 Python 脚本</td>
      <td>掌握 Tensor API 算子工程的完整结构</td>
    </tr>
    <tr>
      <td><a href="#3-tensor-api-矩阵计算流程">3. 矩阵计算流程</a></td>
      <td>数据通路、分形布局、Atom 构造、模板参数</td>
      <td>理解 Tensor API 核心概念和二级切分层次</td>
    </tr>
    <tr>
      <td><a href="#4-单核版本--基线版本">4. 单核版本</a></td>
      <td>基线 kernel 实现、HardEvent 同步机制</td>
      <td>能独立写出单核 Tensor API 矩阵乘法 kernel</td>
    </tr>
    <tr>
      <td><a href="#5-多核并行优化">5. 多核并行优化</a></td>
      <td>核间切分、blockIdx 映射到输出 tile</td>
      <td>理解核间并行切分和 GM 张量切片方法</td>
    </tr>
    <tr>
      <td><a href="#6-double-buffer-优化">6. Double Buffer 优化</a></td>
      <td>双缓冲流水化、多事件 ID 管理</td>
      <td>能通过双缓冲让搬运和计算重叠执行</td>
    </tr>
    <tr>
      <td><a href="#7-性能对比">7. 性能对比</a></td>
      <td>三版本耗时分析与加速比，msprof 采集方法</td>
      <td>能使用 profiling 工具对比优化收益</td>
    </tr>
    <tr>
      <td><a href="#8-课后练习与章节实践">8. 课后练习与章节实践</a></td>
      <td>概念练习 + stepK 大包搬运编程实践</td>
      <td>能独立实现 stepK 优化，减少 GM→L1 搬运次数</td>
    </tr>
  </tbody>
</table>

### 运行环境与硬件说明

- 本节样例 **仅支持 Ascend 950PR/DT**，请在 CANNLab 950 尝鲜体验环境中运行。
- learning-hub 的 notebook 在线体验环境和 CANNLab 910B/910C 云开发环境 **不支持** 本节样例。


---

# 1. 算子接口分析

本节实现的矩阵乘法只有一步计算：

```text
C = A * B
```

其中 `A`、`B` 为 GM 上的 `half` 类型的输入矩阵，`Mmad` 在 L0C 中使用 `float` 累加，最终通过 `CopyL0C2GM` 转换为 `half` 并写回 GM。

## 1.1 输入输出约定

| 参数 | 取值 |
| --- | --- |
| A shape | `[8192, 8192]` |
| B shape | `[8192, 8192]` |
| C shape | `[8192, 8192]` |
| 输入/输出 dtype | `half` |
| Mmad 累加 dtype | `float`，位于 L0C |
| 平台 | Ascend 950PR/DT |

## 1.2 精度校验策略

输入矩阵 A 和 B 使用 `np.random.uniform(-1, 1)` 生成随机数据，真值通过 `np.matmul` 在 float32 下计算后转为 float16 保存为 `golden.bin`。由于 half 类型精度有限，K=8192 的累加会引入舍入误差，校验脚本使用 `np.isclose` 配合 `atol=1e-3`、`rtol=1e-3` 容差定位差异，而不是精确相等。


---

# 2. Tensor API 矩阵乘法样例工程

### 目录结构介绍

本课程代码按照如下目录结构存放，所有样例均位于 `src/04_05_tensor_api_matmul`。

```text
src/04_05_tensor_api_matmul/
├── scripts/
│   ├── gen_data.py                       # 输入数据生成脚本
│   └── verify_result.py                  # 输出结果校验脚本
├── CMakeLists.txt                        # 编译工程文件
├── run.sh                                # 编译、运行和校验脚本
├── data_utils.h                          # 文件读写工具
├── tensor_mmad_host.h                    # Host 侧公共调用流程
├── tensor_mmad_single_core_kernel.h      # 单核版本 kernel 实现
├── tensor_mmad_multi_core_kernel.h       # 多核版本 kernel 实现
├── tensor_mmad_double_buffer_kernel.h    # double buffer 版本 kernel 实现
├── tensor_mmad_stepK_kernel.h            # stepK 大包搬运练习模板（保留 TODO）
├── tensor_mmad_stepK_answer_kernel.h     # stepK 大包搬运参考实现（可直接编译运行）
├── tensor_mmad_single_core.asc           # 单核基线版本入口
├── tensor_mmad_multi_core.asc            # 多核版本入口
├── tensor_mmad_double_buffer.asc         # 多核 + double buffer 版本入口
└── tensor_mmad_practice.asc              # 课后实践入口（stepK 大包搬运）
```


主要文件说明如下：

- `tensor_mmad_host.h`：封装 Host 侧申请内存、读写文件、启动 kernel 和释放资源的公共流程。
- `tensor_mmad_single_core_kernel.h`：单核基线版本 kernel，完整输出矩阵由 1 个 Cube Core 计算。
- `tensor_mmad_multi_core_kernel.h`：多核版本 kernel，按 `singleM/singleN` 切分输出矩阵并映射到 32 个 Cube Core。
- `tensor_mmad_double_buffer_kernel.h`：double buffer 版本 kernel，在多核切分基础上为 L1/L0 数据搬运配置双缓冲。
- `tensor_mmad_stepK_kernel.h`：stepK 大包搬运练习模板，保留 TODO，供学习者在课后实践中完成核心逻辑。
- `tensor_mmad_stepK_answer_kernel.h`：stepK 大包搬运参考实现，供 `practice` target 直接编译运行和校验。
- `tensor_mmad_single_core.asc`、`tensor_mmad_multi_core.asc`、`tensor_mmad_double_buffer.asc`：分别选择对应模板参数并启动 kernel。
- `tensor_mmad_practice.asc`：课后实践验证入口，使用独立命名空间配置 `stepK=4` 参数并启动 stepK 参考实现。
- `scripts/gen_data.py`：生成随机输入矩阵 A 和 B，并用 numpy 计算真值 `golden.bin`。
- `scripts/verify_result.py`：将 kernel 输出与真值按容差比对，校验精度。

注意：为了便于对比不同优化手段，Host 侧公共逻辑放在 `tensor_mmad_host.h`，每个优化版本的 kernel 代码分别放在对应小节中说明。


### 2.1 CMakeLists.txt 和 run.sh

本节新增 CMakeLists.txt，用于编译四个可执行文件。由于 Tensor API 矩阵计算样例只支持 Ascend 950PR/DT，工程中固定使用对应的架构参数。CMake 通过 `ASCEND_HOME_PATH` 环境变量确定 CANN 包路径。

`run.sh` 封装编译、输入生成、运行和精度校验流程，并在每次运行前清理旧的输入输出目录。通过 `--case` 参数可以选择单独运行某个版本：

| 参数值 | 运行内容 |
| --- | --- |
| `single_core` | 单核基线版本 |
| `multi_core` | 多核版本 |
| `double_buffer` | 多核 + double buffer 版本 |
| `practice` | 课后实践版本 |
| `all` | 依次运行前三个版本 |


In [ ]:
%%writefile src/04_05_tensor_api_matmul/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "Tensor API only supports dav-3510 in this sample")

if(NOT CMAKE_ASC_ARCHITECTURES STREQUAL "dav-3510")
    message(FATAL_ERROR "tensor_api_matmul only supports CMAKE_ASC_ARCHITECTURES=dav-3510")
endif()

find_package(ASC REQUIRED)

project(tensor_api_matmul LANGUAGES ASC CXX)

set(ASCEND_HOME $ENV{ASCEND_HOME_PATH})

set(ASC_INCLUDE_DIR "${ASCEND_HOME}/asc/")
include_directories(
    "${ASC_INCLUDE_DIR}"
)


foreach(target_name IN ITEMS
    tensor_mmad_single_core
    tensor_mmad_multi_core
    tensor_mmad_double_buffer
    tensor_mmad_practice
)
    add_executable(${target_name} ${target_name}.asc)
    target_compile_options(${target_name} PRIVATE
        $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
    )
endforeach()

In [ ]:
%%writefile src/04_05_tensor_api_matmul/run.sh
#!/usr/bin/env bash
set -euo pipefail

npu_arch="dav-3510"
case_name="all"

for arg in "$@"; do
    case "$arg" in
        --npu-arch=*)
            npu_arch="${arg#*=}"
            ;;
        --case=*)
            case_name="${arg#*=}"
            ;;
        -h|--help)
            echo "Usage: bash run.sh [--npu-arch=dav-3510] [--case=single_core|multi_core|double_buffer|practice|all]"
            exit 0
            ;;
    esac
done

if [ "$npu_arch" != "dav-3510" ]; then
    echo "Tensor API matrix sample only supports --npu-arch=dav-3510"
    exit 1
fi

if [ -z "${ASCEND_HOME_PATH:-}" ]; then
    echo "Please set ASCEND_HOME_PATH before building."
    exit 1
fi

if [ -f "${ASCEND_HOME_PATH}/set_env.sh" ]; then
    # shellcheck disable=SC1091
    source "${ASCEND_HOME_PATH}/set_env.sh"
fi

rm -rf build_out input output
cmake -S . -B build_out -DCMAKE_ASC_ARCHITECTURES="$npu_arch"
python3 scripts/gen_data.py
mkdir -p output

run_one() {
    local name="$1"
    local binary="$2"
    local out_file="output/${name}.bin"
    echo "[BUILD] ${binary}"
    cmake --build build_out -j"$(nproc)" --target "${binary}"
    echo "[RUN] ${name}"
    ./build_out/${binary}
    python3 scripts/verify_result.py "${out_file}"
}

case "$case_name" in
    single_core)
        run_one single_core tensor_mmad_single_core
        ;;
    multi_core)
        run_one multi_core tensor_mmad_multi_core
        ;;
    double_buffer)
        run_one double_buffer tensor_mmad_double_buffer
        ;;
    practice)
        run_one practice tensor_mmad_practice
        ;;
    all)
        run_one single_core tensor_mmad_single_core
        run_one multi_core tensor_mmad_multi_core
        run_one double_buffer tensor_mmad_double_buffer
        ;;
    *)
        echo "Unsupported case: ${case_name}"
        echo "Use single_core, multi_core, double_buffer, practice, or all."
        exit 1
        ;;
esac


### 2.2 Host 侧公共代码和数据读写工具

`tensor_mmad_host.h` 负责 ACL 初始化、Host/Device 内存申请、输入输出拷贝和 kernel 启动回调；`data_utils.h` 用于从 `.bin` 文件中读取输入数据，并将输出结果写回文件。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/data_utils.h
#ifndef TENSOR_API_MATMUL_DATA_UTILS_H
#define TENSOR_API_MATMUL_DATA_UTILS_H

#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>

#include <cstdio>
#include <fstream>
#include <string>

#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR] " fmt "\n", ##args)

inline bool ReadFile(const std::string &filePath, size_t &fileSize, void *buffer, size_t bufferSize)
{
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file: %s", filePath.c_str());
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file", filePath.c_str());
        return false;
    }

    std::ifstream file(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    std::filebuf *buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0: %s", filePath.c_str());
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size: %s", filePath.c_str());
        return false;
    }

    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char *>(buffer), size);
    fileSize = size;
    return true;
}

inline bool WriteFile(const std::string &filePath, const void *buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("write file failed, buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("write file failed: %s", filePath.c_str());
        return false;
    }
    return true;
}

#endif


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_host.h
#ifndef TENSOR_MMAD_HOST_H
#define TENSOR_MMAD_HOST_H

#include "acl/acl.h"
#include "data_utils.h"

#include <cstdint>
#include <cstdio>
#include <string>

#define CHECK_ACL_RET(expr)                                      \
    do {                                                         \
        aclError ret = (expr);                                   \
        if (ret != ACL_SUCCESS) {                                \
            ERROR_LOG("%s failed, ret = %d", #expr, ret);        \
            return 1;                                            \
        }                                                        \
    } while (0)

template <typename LaunchFunc>
int RunMatmulHost(size_t inputSize, size_t outputSize, const char *outputPath, LaunchFunc launch)
{
    CHECK_ACL_RET(aclInit(nullptr));
    int32_t deviceId = 0;
    CHECK_ACL_RET(aclrtSetDevice(deviceId));

    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream));

    uint8_t *xHost = nullptr;
    uint8_t *yHost = nullptr;
    uint8_t *zHost = nullptr;
    half *xDevice = nullptr;
    half *yDevice = nullptr;
    half *zDevice = nullptr;

    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&xHost), inputSize));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&yHost), inputSize));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&zHost), outputSize));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&xDevice), inputSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&yDevice), inputSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&zDevice), outputSize, ACL_MEM_MALLOC_HUGE_FIRST));

    size_t fileSize = inputSize;
    if (!ReadFile("./input/input_x.bin", fileSize, xHost, inputSize)) {
        return 1;
    }
    fileSize = inputSize;
    if (!ReadFile("./input/input_y.bin", fileSize, yHost, inputSize)) {
        return 1;
    }

    CHECK_ACL_RET(aclrtMemcpy(xDevice, inputSize, xHost, inputSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(yDevice, inputSize, yHost, inputSize, ACL_MEMCPY_HOST_TO_DEVICE));

    launch(xDevice, yDevice, zDevice, stream);
    CHECK_ACL_RET(aclrtSynchronizeStream(stream));

    CHECK_ACL_RET(aclrtMemcpy(zHost, outputSize, zDevice, outputSize, ACL_MEMCPY_DEVICE_TO_HOST));
    if (!WriteFile(outputPath, zHost, outputSize)) {
        return 1;
    }

    (void)aclrtFree(xDevice);
    (void)aclrtFree(yDevice);
    (void)aclrtFree(zDevice);
    (void)aclrtFreeHost(xHost);
    (void)aclrtFreeHost(yHost);
    (void)aclrtFreeHost(zHost);
    (void)aclrtDestroyStream(stream);
    (void)aclrtResetDevice(deviceId);
    (void)aclFinalize();
    return 0;
}

#endif


### 2.3 Python 输入生成脚本和精度校验脚本

Python 脚本统一放在 `scripts` 目录下。`gen_data.py` 生成随机输入矩阵 A 和 B 并保存到 input 目录，同时用 `np.matmul` 计算真值保存到 `output/golden.bin`；`verify_result.py` 将 kernel 输出与真值按容差比对，校验精度。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/scripts/gen_data.py
import os

import numpy as np


M = 8192
N = 8192
K = 8192


def main():
    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    np.random.seed(0)
    a = np.random.uniform(-1, 1, (M, K)).astype(np.float16)
    b = np.random.uniform(-1, 1, (K, N)).astype(np.float16)

    a.tofile("input/input_x.bin")
    b.T.copy().tofile("input/input_y.bin")

    golden = np.matmul(a.astype(np.float32), b.astype(np.float32)).astype(np.float16)
    golden.tofile("output/golden.bin")

    print(f"generated A[{M},{K}] and transposed B file for logical B[{K},{N}]")
    print(f"golden output saved to output/golden.bin")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/04_05_tensor_api_matmul/scripts/verify_result.py
import sys

import numpy as np


M = 8192
N = 8192
K = 8192
ATOL = 1e-3
RTOL = 1e-3


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 verify_result.py output/<case>.bin")

    output_path = sys.argv[1]
    golden_path = "output/golden.bin"

    output = np.fromfile(output_path, dtype=np.float16)
    golden = np.fromfile(golden_path, dtype=np.float16)

    expected_size = M * N
    if output.size != expected_size:
        raise SystemExit(f"invalid output size: {output.size}, expected {expected_size}")
    if golden.size != expected_size:
        raise SystemExit(f"invalid golden size: {golden.size}, expected {expected_size}")

    failed = ~np.isclose(output, golden, atol=ATOL, rtol=RTOL)
    if np.any(failed):
        diff = np.abs(output.astype(np.float32) - golden.astype(np.float32))
        diff_idx = np.flatnonzero(failed)
        first = int(diff_idx[0])
        raise SystemExit(
            f"verify failed at flat index {first}: "
            f"got {output[first]}, golden {golden[first]}, abs diff {diff[first]}"
        )

    print(f"verify passed: {output_path}")


if __name__ == "__main__":
    main()


### 运行前检查

运行样例前确认 `ASCEND_HOME_PATH` 已设置，并切换到 `src/04_05_tensor_api_matmul` 目录：

```bash
export ASCEND_HOME_PATH=/usr/local/Ascend/ascend-toolkit/latest
cd src/04_05_tensor_api_matmul && bash run.sh --npu-arch=dav-3510 --case=multi_core
```


---

# 3. Tensor API 矩阵计算流程

Tensor API 版本的矩阵乘法按照基本块循环计算，每组 A/B 基本块主要经过以下流水线阶段：

| 阶段 | Operation | 功能 |
| --- | --- | --- |
| 输入数据搬入 | `CopyGM2L1` | 将 A、B 的当前基本块从 GM 搬入 L1，并完成相应格式转换 |
| 搬入 L0 | `CopyL12L0A` / `CopyL12L0B` | 将 L1 中的 A、B 基本块搬入 L0A/L0B |
| 矩阵计算 | `Mmad` | 在 Cube 计算单元上完成 `A * B`，结果累加到 L0C |
| 输出搬出 | `CopyL0C2GM` | 将 L0C 上的 `float` 结果转换为 `half` 并写回 GM |

整体数据流为：从 GM 中读取 `A` 的当前基本块和 `B` 的当前基本块，搬入 L1 后转换为对应格式，再搬入 L0A/L0B 参与 Mmad 计算。沿 K 方向的多个分块累加完成后，再将 L0C 结果写回 GM。

本节前三个版本（单核、多核、double buffer）的样例每次只搬运一组 A/B 基本块，其中 A 基本块大小为 `baseM * baseK`，B 基本块大小为 `baseK * baseN`，不使用大包搬运。课后实践版本在此基础上引入 `stepK` 参数，一次搬运多组基本块以优化 GM 访问效率。

![Tensor API pipeline](images/04_05_tensor_api_matmul/tensor_api_pipeline.png)


首次阅读 Tensor API 代码时，先理解以下概念：

| 概念 | 作用 |
| --- | --- |
| `MakeMemPtr` | 把裸指针包装成 Tensor API 可识别的内存指针 |
| `MakeTensor` | 用内存指针和布局构造张量视图，本身不搬运数据 |
| `MakeFrameLayout` | 描述矩阵在某个存储层级中的逻辑 shape 和物理排布 |
| `MakeCoord` / `MakeShape` | 构造坐标和形状，用于 `Slice` 取子矩阵 |
| `Slice` | 从一个大 Tensor 中取出子矩阵 |

### LayoutPattern 与存储层级

不同存储层级要求不同的 LayoutPattern。本样例中用到的布局模式如下：

| LayoutPattern | 含义 | 使用场景 |
| --- | --- | --- |
| `NDLayoutPtn` | 行主序（扩展嵌套布局） | GM 上的矩阵 A、C（推荐用法） |
| `DNLayoutPtn` | 列主序（扩展嵌套布局） | GM 上的矩阵 B（转置存储，推荐用法） |
| `NDLayoutPtn` | 行主序（扁平布局） | GM 上的矩阵 A、C（简化写法） |
| `DNLayoutPtn` | 列主序（扁平布局） | GM 上的矩阵 B（转置存储，简化写法） |
| `NZLayoutPtn` | 内层 Z 形、外层 N 形分形 | L1/L0 上的矩阵 A、L0C 上的结果 C |
| `ZNLayoutPtn` | 内层 N 形、外层 Z 形分形 | L1/L0 上的矩阵 B |

> **说明**：`NDLayoutPtn`/`DNLayoutPtn` 与 `NDLayoutPtn`/`DNLayoutPtn` 都描述 ND/DN 排布，区别在于内部 Layout 结构的嵌套层次不同。本课程统一使用 `NDLayoutPtn`/`DNLayoutPtn`，与 asc-devkit 参考样例保持一致。

NZ 和 ZN 是 Cube 计算单元要求的分形格式。`half` 类型下，内层分形块为 `16 × 16`（16 行 × 16 列），NZ 表示内层按 Z 字形（行优先）排列、外层按 N 字形（列优先）排列；ZN 则相反。`CopyGM2L1` 搬运时会自动完成从 ND/DN 到 NZ/ZN 的格式转换。

![LayoutPattern 分形布局转换](images/04_05_tensor_api_matmul/layout_conversion.png)


### Atom 构造与 `with()` 参数绑定

Tensor API 的搬运和计算都遵循 **"构造 Atom → 绑定参数 → 调用"** 的三步模式。在 kernel 代码中会看到如下写法：

```cpp
// 第 1 步：在循环外构造 Atom（只需一次）
auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
auto mmadAtom       = MakeMmad(MmadOperation{}, MmadTraitDefault{});

// 第 2 步：在循环内绑定运行时参数并调用
Copy(copyGM2L1Atom, l1ATensor, gmATensor.Slice(...));
Mmad(mmadAtom.with(MmadParams{m, n, k, 0, true}), l0CTensor, l0ATensor, l0BTensor);
```

**Operation（操作类型）**

Operation 描述搬运或计算的操作种类，是一个空标签类型：

| Operation | 含义 |
| --- | --- |
| `CopyGM2L1` | GM → L1 搬运 |
| `CopyL12L0A` / `CopyL12L0B` | L1 → L0A / L0B 搬运 |
| `CopyL0C2GM` | L0C → GM 搬运（通过 Fixpipe 固定流水线） |
| `MmadOperation` | 矩阵乘加计算 |

**Trait（特征配置）**

Trait 描述操作的附加属性，在编译期确定，编译器据此选择底层指令。每种 Operation 都有对应的 `TraitDefault`：

| Trait 默认值 | 适用 Operation | 可配置项（示例） |
| --- | --- | --- |
| `CopyGM2L1TraitDefault` | `CopyGM2L1` | 格式转换模式（ND→NZ 等） |
| `CopyL0C2GMTraitDefault` | `CopyL0C2GM` | `enableRelu`、`roundMode` |
| `MmadTraitDefault` | `MmadOperation` | `disableGemv`、`mmadType`（普通/MX） |

当默认 Trait 不满足需求时（如开启 MX 模式），可传入自定义 Trait 替代 `TraitDefault`，后续课程会涉及。

**`MakeCopy` / `MakeMmad` 构造 Atom**

`MakeCopy(Operation{}, TraitDefault{})` 将 Operation 和 Trait 组合为一个 **Atom 对象**。Atom 在循环外构造一次即可复用，避免循环内重复构造。

**`.with()` 绑定运行时参数**

`.with(params)` 将运行时可变的参数绑定到 Atom，返回一个携带参数的新对象。`MmadParams` 包含以下字段：

| 字段 | 类型 | 含义 |
| --- | --- | --- |
| `m` | `uint16_t` | 左矩阵 A 的高度（结果 C 的高度） |
| `n` | `uint16_t` | 右矩阵 B 的宽度（结果 C 的宽度） |
| `k` | `uint16_t` | A 的宽度 / B 的高度（本次累加的 K 维长度） |
| `unitFlag` | `uint8_t` | Mmad 与 Fixpipe（L0C 搬出）细粒度并行控制：`0`=不使能，`2`=使能不复位，`3`=使能且复位。本课程基础版本不使用该特性（设为 `0`），后续高级课程会涉及 |
| `cmatrixInitVal` | `bool` | 不传 bias 时：`true`=初始化 L0C 为零，`false`=基于 C 原值累加。K 方向首次分块设为 `true`，后续设为 `false` |

这些值每次 K 循环都可能变化。`MmadParams` 支持聚合初始化：`MmadParams params{m, n, k, 0, (ki == 0)}`。Copy 如果不需要额外运行时参数则直接使用 Atom 调用。

**调用**

```cpp
Copy(atom, dstTensor, srcTensor);                          // 搬运
Copy(atom, dstTensor, srcTensor, quantTensor);             // 带随路量化
Mmad(atom.with(params), dstTensor, fmTensor, filterTensor); // 矩阵乘加
Mmad(atom.with(params), dstTensor, fmTensor, filterTensor, biasTensor); // 带 bias
```

这种设计的优势在于：编译期固定的 Trait 让编译器选择最优指令路径，运行期可变的 Params 通过 `.with()` 灵活传入，两者解耦且 Atom 可复用。



### 二级切分层次

```text
完整矩阵 [M, N] × K
  ├─ 核间切分：singleM × singleN，每个 Cube Core 负责一个 tile
  └─ 核内切分：baseM × baseN × baseK，每次 Mmad 计算一个基本块
```

循环次数由切分参数推导：`mLoop = singleM / baseM`，`nLoop = singleN / baseN`，`kLoop = singleK / baseK`。

![切分层次](images/04_05_tensor_api_matmul/tiling_hierarchy.png)


### singleK 参数说明

`singleK` 描述一个 Cube Core 负责的 K 方向范围。在本节单核、多核和 double buffer 版本中，`singleK` 均设置为等于 `K`（8192），即每个 Cube Core 沿 K 方向处理完整的矩阵宽度，不在 K 方向做核间切分。

这样设计的目的是简化核间映射逻辑：每个 Cube Core 拿到的是完整的 K 列数据，只需按 M/N 方向切分输出区域。如果问题规模远超本节示例（例如 K=32768），可以考虑在 K 方向也引入核间切分，此时 `mCoreIdx` 的映射需要包含 `kCoreLoop` 维度，形成 `blockIdx` → `(mCoreIdx, nCoreIdx, kCoreIdx)` 的三维映射。本节暂不涉及这种情况。


---

# 4. 单核版本 —— 基线版本

从这一节开始，我们按"单核基线版本 → 多核并行 → double buffer"的路线逐步优化。三个版本的完整问题规模、输入输出类型和基本块计算路径保持一致。

![Optimization route](images/04_05_tensor_api_matmul/optimization_route.png)

首先实现单核版本作为后续优化的基线。单核版本将完整 `[8192,8192]` 输出交给一个 Cube Core 计算，并在该 Cube Core 内部按 `baseM/baseN/baseK` 做基本块循环。

该版本并行度最低，但代码路径最直接，适合用来确认 Tensor API 数据搬运、K 方向分块累加和 L0C 初始化逻辑是否正确。后续优化版本都保持相同的输入输出约定，再比较性能差异。

> **搬运策略说明**：本课程单核版本在 K 循环内逐块搬运 GM→L1（每次搬运一个 `baseK` 宽度的基本块），目的是为后续 stepK 大包搬运优化做铺垫——先建立"每次搬运一块"的基线，再在课后实践中体会"一次搬运多块"的优化收益。asc-devkit 入门样例 `matmul_tensor_api` 则采用另一种策略：在循环外一次性搬运整个 K 范围到 L1，K 循环内只做 L1→L0 和 Mmad。两种策略各有适用场景，本课程选择前者是为了教学递进。

单核版本的主要特点如下：

| 维度 | 说明 |
| --- | --- |
| 解决的问题 | 建立可运行、可校验的最小矩阵计算路径 |
| 切分方式 | `singleM=8192, singleN=8192, singleK=8192`，只启动 1 个 Cube Core 任务 |
| 优点 | 代码路径简单，便于定位 Tensor、Layout、Mmad 和 Copy 的使用问题 |
| 局限 | Ascend 950PR/DT 的多数 Cube Core 处于空闲状态，总耗时较长 |

写入下面的 kernel 和入口文件后，运行脚本验证计算结果：


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_single_core_kernel.h
#ifndef TENSOR_MMAD_SINGLE_CORE_KERNEL_H
#define TENSOR_MMAD_SINGLE_CORE_KERNEL_H

#include "basic_api/kernel_operator_block_sync_intf.h"
#include "tensor_api/tensor.h"

template <
    int32_t M, int32_t N, int32_t K,
    int32_t singleM, int32_t singleN, int32_t singleK,
    int32_t baseM, int32_t baseN, int32_t baseK>
__cube__ __global__ void TensorMmadSingleCoreKernel(__gm__ half *x, __gm__ half *y, __gm__ half *z)
{
    using namespace AscendC::Te;
    static_assert(M == singleM && N == singleN && K == singleK);
    static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);

    constexpr uint32_t mLoop = singleM / baseM;
    constexpr uint32_t nLoop = singleN / baseN;
    constexpr uint32_t kLoop = singleK / baseK;

    auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
    auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
    auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));

    __cbuf__ half l1ABuf[baseM * baseK];
    __cbuf__ half l1BBuf[baseK * baseN];
    __ca__ half l0ABuf[baseM * baseK];
    __cb__ half l0BBuf[baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
    auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
    auto copyL12L0BAtom = MakeCopy(CopyL12L0B{}, CopyL12L0BTraitDefault{});
    auto copyL0C2GMAtom = MakeCopy(CopyL0C2GM{}, CopyL0C2GMTraitDefault{});
    auto mmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

    auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

    constexpr uint32_t L1_EVENT_ID = 0;
    constexpr uint32_t L0_EVENT_ID = 1;
    constexpr uint32_t L0C_EVENT_ID = 2;
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);

    for (uint32_t mi = 0; mi < mLoop; ++mi) {
        for (uint32_t ni = 0; ni < nLoop; ++ni) {
            for (uint32_t ki = 0; ki < kLoop; ++ki) {
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                Copy(copyGM2L1Atom, l1ATensor, gmATensor.Slice(MakeCoord(mi * baseM, ki * baseK), MakeShape(baseM, baseK)));
                Copy(copyGM2L1Atom, l1BTensor, gmBTensor.Slice(MakeCoord(ki * baseK, ni * baseN), MakeShape(baseK, baseN)));
                AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);

                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
                Copy(copyL12L0AAtom, l0ATensor, l1ATensor);
                Copy(copyL12L0BAtom, l0BTensor, l1BTensor);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);

                if (ki == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
                }
                MmadParams params{baseM, baseN, baseK, 0, (ki == 0)};
                Mmad(mmadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
                if (ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
            }

            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
            Copy(copyL0C2GMAtom, gmCTensor.Slice(MakeCoord(mi * baseM, ni * baseN), MakeShape(baseM, baseN)), l0CTensor);
            AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
        }
    }

    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
}

#endif


### 4.1 单核 kernel 代码讲解

上面的 `TensorMmadSingleCoreKernel` 按功能分为以下几个部分：

#### 4.1.1 模板参数与静态检查

```cpp
static_assert(M == singleM && N == singleN && K == singleK);
static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);
```

单核版本要求 `singleM = M`、`singleN = N`，即不分核，所有计算由 1 个 Cube Core 完成。第二行确保基本块能整除 tile。

#### 4.1.2 GM 张量构造

```cpp
auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));
```

三个 GM 张量分别描述输入 A（行主序 `NDLayoutPtn`）、输入 B（列主序 `DNLayoutPtn`，即转置存储）和输出 C（行主序 `NDLayoutPtn`）。`MakeTensor` 只构造视图，不搬运数据。

**为什么 B 矩阵使用转置存储（`DNLayoutPtn`）？**

在矩阵乘法 `C = A × B` 中，A 的逻辑 shape 为 `[M, K]`，B 的逻辑 shape 为 `[K, N]`。本课程将 B 在 GM 中以列主序存储，即在物理内存中 B 的第一行（对应 K 方向）连续排列。这样做的好处是：当沿 K 方向切分 B 时，每个 `baseK × baseN` 的基本块在物理内存中是连续的，有利于 `CopyGM2L1` 高效搬运。`gen_data.py` 中通过 `b.T.copy().tofile()` 将 B 转置后保存，与本课程的 `DNLayoutPtn` 布局对应。

> **与参考样例的差异**：asc-devkit 中的入门样例 `matmul_tensor_api` 使用 `NDLayoutPtn` 存储所有 GM 矩阵（不转置 B），此时 L1B 使用 `NZLayoutPtn`。本课程选择转置存储 B，因此 L1B 使用 `ZNLayoutPtn`。两种方式都是合法的，关键在于 GM 布局与 L1 布局的转换关系需要匹配。

#### 4.1.3 Buffer 声明与原子对象构造

```cpp
__cbuf__ half l1ABuf[baseM * baseK];   // L1 上的 A 缓冲
__ca__ half l0ABuf[baseM * baseK];     // L0A 上的 A 缓冲
__cc__ float l0CBuf[baseM * baseN];    // L0C 上的结果缓冲（float 累加）
```

`__cbuf__`、`__ca__`、`__cb__`、`__cc__` 是 Ascend C 的存储空间修饰符，分别对应 L1、L0A、L0B、L0C Buffer。L0C 用 `float` 类型是因为 Mmad 在 L0C 中以 float 累加。

#### 4.1.4 三层循环与同步

```cpp
for (mi...) for (ni...) for (ki...) {
    // GM -> L1
    Copy(copyGM2L1Atom, l1ATensor, gmATensor.Slice(...));
    // L1 -> L0
    Copy(copyL12L0AAtom, l0ATensor, l1ATensor);
    // Mmad
    Mmad(mmadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
}
// L0C -> GM
Copy(copyL0C2GMAtom, gmCTensor.Slice(...), l0CTensor);
```

最外层 `mi`、`ni` 遍历输出基本块，内层 `ki` 沿 K 方向累加。每个输出基本块的第一个 K 分块（`ki == 0`）设置 `cmatrixInitVal = true` 初始化 L0C 为零，后续分块累加。K 方向循环结束后，通过 `CopyL0C2GM` 将 L0C 的 float 结果转为 half 写回 GM。

---

### 4.2 单核版本的 HardEvent 同步分析

流水同步核心是保证每个指令依赖的数据已经准备完毕：

![单核版本流水时序](images/04_05_tensor_api_matmul/single_core_pipeline.png)

HardEvent 的命名规则是 `X_Y`，表示 X 阶段完成后允许 Y 阶段开始。对应到代码，各同步点分别保护以下数据依赖：

| 同步点 | 指令顺序 | 保护的数据 | 如果缺失可能发生什么 |
| --- | --- | --- | --- |
| `MTE1_MTE2` | `Copy(L1→L0)` → 下一轮 `Copy(GM→L1)` | L1 中的 A/B 缓冲 | 新一轮 GM→L1 可能覆盖尚未搬运到 L0 的数据 |
| `MTE2_MTE1` | `Copy(GM→L1)` → `Copy(L1→L0)` | L1 中刚搬入的矩阵分块 | `Copy(L1→L0)` 可能读取尚未搬完的数据 |
| `M_MTE1` | `Mmad` → 下一轮 `Copy(L1→L0)` | L0A/L0B 中的输入缓冲 | 新一轮 L1→L0 可能覆盖 Mmad 正在使用的输入 |
| `MTE1_M` | `Copy(L1→L0)` → `Mmad` | L0A/L0B 中的分形数据 | `Mmad` 可能提前读取 L0A/L0B |
| `M_FIX` | `Mmad` → `Copy(L0C→GM)` | L0C 中的矩阵乘结果 | `CopyL0C2GM` 可能把未完成的 L0C 写回 GM |
| `FIX_M` | `Copy(L0C→GM)` → 下一输出块首轮 `Mmad` | L0C 中的累加结果 | 下一输出块的 Mmad 可能使用上一块未搬完的 L0C |

### SetFlag / WaitFlag 配对说明

使用时需要配对调用：

- **SetFlag** 在 X 阶段的指令**发出后**调用，设置硬件信号表示「X 阶段已发起，Y 阶段可以等待后继续」。注意 SetFlag 是在指令发出时设置信号，而非等待指令执行完毕。
- **WaitFlag** 在 Y 阶段**开始前**调用，阻塞直到 X 阶段的信号到达，表示「等待 X 阶段完成」。

以 L1→L0 搬运到 Mmad 的流程为例，代码中的配对关系如下：

```text
WaitFlag<M_MTE1>   ← 等待上一轮 Mmad 完成后 L0 缓冲可用
Copy(L1→L0A/L0B)   ← 执行 L1→L0 搬运
SetFlag<MTE1_M>    ← 通知 Mmad：L0 数据已准备好
WaitFlag<MTE1_M>   ← Mmad 等待 L0 数据就绪
Mmad(...)          ← 执行矩阵计算
SetFlag<M_MTE1>    ← 通知 MTE1：L0 缓冲可复用
```

注意 `SetFlag` 和 `WaitFlag` 使用**相同**的 HardEvent 名称和 Event ID：`SetFlag` 在当前流水阶段完成后发出信号，`WaitFlag` 在依赖该信号的阶段开始前阻塞。例如 `SetFlag<MTE1_MTE2>(L1_EVENT_ID)` 和 `WaitFlag<MTE1_MTE2>(L1_EVENT_ID)` 配对，前者在 L1→L0 完成后通知 L1 缓冲可复用，后者在下一轮 GM→L1 前等待该信号。理解这个配对关系是正确编写同步代码的关键。

在实际代码中可以看到：L1→L0 搬运完成后执行 `SetFlag<MTE1_MTE2>` 通知 GM→L1 可以复用 L1 缓冲；同时执行 `SetFlag<MTE1_M>` 通知 Mmad 可以开始计算。这两个通知分别解除不同阶段的等待。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_single_core.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 */

#include "tensor_mmad_host.h"
#include "tensor_mmad_single_core_kernel.h"

namespace SingleCoreConfig {
constexpr int32_t M = 8192;
constexpr int32_t N = 8192;
constexpr int32_t K = 8192;
constexpr int32_t singleM = 8192;
constexpr int32_t singleN = 8192;
constexpr int32_t singleK = 8192;
constexpr int32_t baseM = 128;
constexpr int32_t baseN = 256;
constexpr int32_t baseK = 64;
constexpr uint32_t NUM_BLOCKS = 1;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace SingleCoreConfig;

    constexpr size_t inputSize = static_cast<size_t>(M) * K * sizeof(half);
    constexpr size_t outputSize = static_cast<size_t>(M) * N * sizeof(half);

    auto launch = [](half *x, half *y, half *z, aclrtStream stream) {
        TensorMmadSingleCoreKernel<M, N, K, singleM, singleN, singleK, baseM, baseN, baseK>
            <<<NUM_BLOCKS, nullptr, stream>>>(x, y, z);
    };
    return RunMatmulHost(inputSize, outputSize, "./output/single_core.bin", launch);
}


In [ ]:
!cd src/04_05_tensor_api_matmul && bash run.sh --npu-arch=dav-3510 --case=single_core

---

# 5. 多核并行优化

单核版本没有用满 Cube Core 资源。接下来保持 `M/N/K = 8192` 不变，将输出矩阵按 `singleM = 1024`、`singleN = 2048` 切为 `8 * 4 = 32` 个核心任务。

Ascend 950PR/DT 一共有 32 个 Cube Core，因此这里让每个核心负责一个输出 tile，每个核心内部仍然按 `baseM/baseN/baseK = 128/256/64` 做基本块计算。每个输出 tile 之间没有数据依赖，且写回 GM 的区域互不重叠，因此适合直接按照 `GetBlockIdx()` 做任务映射。

### blockIdx 到 tile 的映射

```text
blockIdx = mCoreIdx + nCoreIdx * mCoreLoop

其中：
  mCoreLoop = M / singleM = 8192 / 1024 = 8
  nCoreLoop = N / singleN = 8192 / 2048 = 4
  总核数 = mCoreLoop * nCoreLoop = 32

反解：
  mCoreIdx = blockIdx % mCoreLoop
  nCoreIdx = blockIdx / mCoreLoop
```

每个 Cube Core 通过 `GetBlockIdx()` 获取自己的编号，反解出 `(mCoreIdx, nCoreIdx)`，然后通过 `Slice` 从 GM 中切出自己负责的 A、B、C 子矩阵。

多核版本的主要优化点如下：

| 维度 | 说明 |
| --- | --- |
| 解决的问题 | 单核版本只有 1 个 Cube Core 工作，硬件利用率低 |
| 切分方式 | `M` 方向 8 份，`N` 方向 4 份，总共 32 个 Cube Core 任务 |
| 映射方式 | `blockIdx` 映射到 `(mCoreIdx, nCoreIdx)`，每个 Cube Core 处理一个输出 tile |
| 预期收益 | 多个 Cube Core 并行计算，缩短总耗时 |

![多核切分](images/04_05_tensor_api_matmul/multi_core_tiling.png)

同样先写入 kernel 和入口文件，再运行验证：


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_multi_core_kernel.h
#ifndef TENSOR_MMAD_MULTI_CORE_KERNEL_H
#define TENSOR_MMAD_MULTI_CORE_KERNEL_H

#include "basic_api/kernel_operator_block_sync_intf.h"
#include "tensor_api/tensor.h"

template <
    int32_t M, int32_t N, int32_t K,
    int32_t singleM, int32_t singleN, int32_t singleK,
    int32_t baseM, int32_t baseN, int32_t baseK>
__cube__ __global__ void TensorMmadMultiCoreKernel(__gm__ half *x, __gm__ half *y, __gm__ half *z)
{
    using namespace AscendC::Te;
    static_assert(M % singleM == 0 && N % singleN == 0 && K % singleK == 0);
    static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);

    constexpr uint32_t mCoreLoop = M / singleM;
    constexpr uint32_t nCoreLoop = N / singleN;
    constexpr uint32_t mLoop = singleM / baseM;
    constexpr uint32_t nLoop = singleN / baseN;
    constexpr uint32_t kLoop = singleK / baseK;

    uint32_t blockIdx = AscendC::GetBlockIdx();
    uint32_t mCoreIdx = blockIdx % mCoreLoop;
    uint32_t nCoreIdx = blockIdx / mCoreLoop;

    auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
    auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
    auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));

    auto gmASingle = gmATensor.Slice(MakeCoord(mCoreIdx * singleM, 0), MakeShape(singleM, singleK));
    auto gmBSingle = gmBTensor.Slice(MakeCoord(0, nCoreIdx * singleN), MakeShape(singleK, singleN));
    auto gmCSingle = gmCTensor.Slice(MakeCoord(mCoreIdx * singleM, nCoreIdx * singleN), MakeShape(singleM, singleN));

    __cbuf__ half l1ABuf[baseM * baseK];
    __cbuf__ half l1BBuf[baseK * baseN];
    __ca__ half l0ABuf[baseM * baseK];
    __cb__ half l0BBuf[baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
    auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
    auto copyL12L0BAtom = MakeCopy(CopyL12L0B{}, CopyL12L0BTraitDefault{});
    auto copyL0C2GMAtom = MakeCopy(CopyL0C2GM{}, CopyL0C2GMTraitDefault{});
    auto mmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

    auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

    constexpr uint32_t L1_EVENT_ID = 0;
    constexpr uint32_t L0_EVENT_ID = 1;
    constexpr uint32_t L0C_EVENT_ID = 2;
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);

    for (uint32_t mi = 0; mi < mLoop; ++mi) {
        for (uint32_t ni = 0; ni < nLoop; ++ni) {
            for (uint32_t ki = 0; ki < kLoop; ++ki) {
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                Copy(copyGM2L1Atom, l1ATensor, gmASingle.Slice(MakeCoord(mi * baseM, ki * baseK), MakeShape(baseM, baseK)));
                Copy(copyGM2L1Atom, l1BTensor, gmBSingle.Slice(MakeCoord(ki * baseK, ni * baseN), MakeShape(baseK, baseN)));
                AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);

                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
                Copy(copyL12L0AAtom, l0ATensor, l1ATensor);
                Copy(copyL12L0BAtom, l0BTensor, l1BTensor);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);

                if (ki == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
                }
                MmadParams params{baseM, baseN, baseK, 0, (ki == 0)};
                Mmad(mmadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
                if (ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
            }

            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
            Copy(copyL0C2GMAtom, gmCSingle.Slice(MakeCoord(mi * baseM, ni * baseN), MakeShape(baseM, baseN)), l0CTensor);
            AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
        }
    }

    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
}

#endif


### 5.1 多核 kernel 与单核 kernel 的差异

多核版本 `TensorMmadMultiCoreKernel` 在单核版本基础上增加了核间切分逻辑，核心差异在以下几处：

#### 5.1.1 核间映射

```cpp
uint32_t blockIdx = AscendC::GetBlockIdx();
uint32_t mCoreIdx = blockIdx % mCoreLoop;
uint32_t nCoreIdx = blockIdx / mCoreLoop;
```

每个核根据自己的 `blockIdx` 计算出负责的 tile 坐标。

#### 5.1.2 GM 张量切片

```cpp
auto gmASingle = gmATensor.Slice(MakeCoord(mCoreIdx * singleM, 0), MakeShape(singleM, singleK));
auto gmBSingle = gmBTensor.Slice(MakeCoord(0, nCoreIdx * singleN), MakeShape(singleK, singleN));
auto gmCSingle = gmCTensor.Slice(MakeCoord(mCoreIdx * singleM, nCoreIdx * singleN), MakeShape(singleM, singleN));
```

每个核从完整的 GM 张量中切出自己负责的子矩阵。A 沿 M 方向切分（每个核取不同行段），B 沿 N 方向切分（每个核取不同列段），C 对应自己的输出区域。

#### 5.1.3 入口文件的 NUM_BLOCKS

```cpp
constexpr uint32_t NUM_BLOCKS = (M / singleM) * (N / singleN);  // = 32
```

启动 kernel 时传入 32 个 block，硬件会将其分配到 32 个 Cube Core 上并行执行。其余的 L1/L0 Buffer 声明、三层循环和同步逻辑与单核版本完全一致。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_multi_core.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 */

#include "tensor_mmad_host.h"
#include "tensor_mmad_multi_core_kernel.h"

namespace MultiCoreConfig {
constexpr int32_t M = 8192;
constexpr int32_t N = 8192;
constexpr int32_t K = 8192;
constexpr int32_t singleM = 1024;
constexpr int32_t singleN = 2048;
constexpr int32_t singleK = 8192;
constexpr int32_t baseM = 128;
constexpr int32_t baseN = 256;
constexpr int32_t baseK = 64;
constexpr uint32_t NUM_BLOCKS = (M / singleM) * (N / singleN);
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace MultiCoreConfig;

    constexpr size_t inputSize = static_cast<size_t>(M) * K * sizeof(half);
    constexpr size_t outputSize = static_cast<size_t>(M) * N * sizeof(half);

    auto launch = [](half *x, half *y, half *z, aclrtStream stream) {
        TensorMmadMultiCoreKernel<M, N, K, singleM, singleN, singleK, baseM, baseN, baseK>
            <<<NUM_BLOCKS, nullptr, stream>>>(x, y, z);
    };
    return RunMatmulHost(inputSize, outputSize, "./output/multi_core.bin", launch);
}


In [ ]:
!cd src/04_05_tensor_api_matmul && bash run.sh --npu-arch=dav-3510 --case=multi_core

多核优化把输出矩阵拆成多个互不依赖的 tile，让 Ascend 950PR/DT 的 32 个 Cube Core 同时工作。相比单核版本，单个 Cube Core 的工作量变小，但整体并行度提高。性能是否接近理想加速，还会受到 GM 带宽、L1/L0 搬运开销和 tile 尺寸选择的影响。


---

# 6. Double Buffer 优化

多核版本提高了并行度，但单个 Cube Core 内部仍然存在搬运和计算之间的等待。每个输出 tile 都需要沿 K 方向循环多次：当前 K 分块搬入 L1/L0 后才能执行 Mmad，Mmad 完成后再进入下一轮搬运。若搬运和计算串行等待较多，Cube Core 利用率上不去。

### 无 double buffer 时的串行等待

```text
K分块0: [搬入L1] → [搬入L0] → [Mmad]
K分块1:                            [搬入L1] → [搬入L0] → [Mmad]
K分块2:                                                       [搬入L1] → ...
```

每个 K 分块必须等前一个分块的 Mmad 完成后才能开始搬入，搬运和计算无法重叠。

### double buffer 时的交替流水

```text
缓冲0: [搬入L1] → [搬入L0] → [Mmad]
缓冲1:       [搬入L1] → [搬入L0] → [Mmad]
缓冲0:                             [搬入L1] → [搬入L0] → [Mmad]
缓冲1:                                       [搬入L1] → ...
```

double buffer 版本在多核切分不变的基础上，为 L1A/L1B 和 L0A/L0B 准备两份输入缓冲。不同 K 分块按缓冲编号交替使用输入缓冲，让搬运和计算有机会重叠。这里仍然保持基本块搬运方式，不引入大包搬运，这样可以更清楚地看到双缓冲本身对代码结构和性能的影响。

double buffer 的主要特点如下：

| 维度 | 说明 |
| --- | --- |
| 解决的问题 | 单个 Cube Core 内部 GM/L1/L0 搬运和 Mmad 之间存在等待 |
| 缓冲方式 | L1A/L1B 和 L0A/L0B 各准备两份，按 K 分块交替使用 |
| 预期收益 | 减少输入缓冲复用等待，让搬运和计算有机会重叠 |
| 代价 | L1/L0 Buffer 占用增加，同步关系更复杂 |
| 适用条件 | 当搬运耗时在总耗时中占比较明显时，收益更容易体现 |


### HardEvent 同步说明

HardEvent 用来表达不同执行单元之间的依赖关系。本样例中用到的 HardEvent 及其含义：

| HardEvent | 含义 | 使用场景 |
| --- | --- | --- |
| `MTE1_MTE2` | L1 缓冲可被 GM 搬入复用 | 上一轮 L1→L0 搬运完成后，允许新一轮 GM→L1 |
| `MTE2_MTE1` | GM 到 L1 搬运完成 | GM→L1 完成后允许 L1→L0 |
| `M_MTE1` | L0 输入缓冲可被 L1→L0 搬运复用 | 上一轮 Mmad 完成后，允许新一轮 L1→L0 |
| `MTE1_M` | L0 输入已准备好供 Mmad 使用 | L1→L0 完成后允许 Mmad |
| `M_FIX` | Mmad 完成后允许 L0C 搬出 | Mmad 完成→允许 CopyL0C2GM 将 L0C 结果搬出到 GM |
| `FIX_M` | L0C 搬出完成后允许下一轮 Mmad | CopyL0C2GM 完成→允许下一输出块的 Mmad 使用 L0C |

double buffer 版本为每个缓冲分配独立的事件 ID（`L1_EVENT_0/1`、`L0_EVENT_0/1`），使两份缓冲的同步互不干扰。

![Double buffer](images/04_05_tensor_api_matmul/double_buffer.png)

写入 kernel 和入口文件后运行验证：


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_double_buffer_kernel.h
#ifndef TENSOR_MMAD_DOUBLE_BUFFER_KERNEL_H
#define TENSOR_MMAD_DOUBLE_BUFFER_KERNEL_H

#include "basic_api/kernel_operator_block_sync_intf.h"
#include "tensor_api/tensor.h"

template <
    int32_t M, int32_t N, int32_t K,
    int32_t singleM, int32_t singleN, int32_t singleK,
    int32_t baseM, int32_t baseN, int32_t baseK>
__cube__ __global__ void TensorMmadDoubleBufferKernel(__gm__ half *x, __gm__ half *y, __gm__ half *z)
{
    using namespace AscendC::Te;
    static_assert(M % singleM == 0 && N % singleN == 0 && K % singleK == 0);
    static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);

    constexpr uint32_t mCoreLoop = M / singleM;
    constexpr uint32_t nCoreLoop = N / singleN;
    constexpr uint32_t mLoop = singleM / baseM;
    constexpr uint32_t nLoop = singleN / baseN;
    constexpr uint32_t kLoop = singleK / baseK;

    uint32_t blockIdx = AscendC::GetBlockIdx();
    uint32_t mCoreIdx = blockIdx % mCoreLoop;
    uint32_t nCoreIdx = blockIdx / mCoreLoop;

    auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
    auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
    auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));

    auto gmASingle = gmATensor.Slice(MakeCoord(mCoreIdx * singleM, 0), MakeShape(singleM, singleK));
    auto gmBSingle = gmBTensor.Slice(MakeCoord(0, nCoreIdx * singleN), MakeShape(singleK, singleN));
    auto gmCSingle = gmCTensor.Slice(MakeCoord(mCoreIdx * singleM, nCoreIdx * singleN), MakeShape(singleM, singleN));

    __cbuf__ half l1ABuf[2 * baseM * baseK];
    __cbuf__ half l1BBuf[2 * baseK * baseN];
    __ca__ half l0ABuf[2 * baseM * baseK];
    __cb__ half l0BBuf[2 * baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
    auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
    auto copyL12L0BAtom = MakeCopy(CopyL12L0B{}, CopyL12L0BTraitDefault{});
    auto copyL0C2GMAtom = MakeCopy(CopyL0C2GM{}, CopyL0C2GMTraitDefault{});
    auto mmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

    constexpr uint32_t L1_EVENT_0 = 0;
    constexpr uint32_t L1_EVENT_1 = 1;
    constexpr uint32_t L0_EVENT_0 = 2;
    constexpr uint32_t L0_EVENT_1 = 3;
    constexpr uint32_t L0C_EVENT_ID = 4;
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_0);
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_1);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_0);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_1);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);

    for (uint32_t mi = 0; mi < mLoop; ++mi) {
        for (uint32_t ni = 0; ni < nLoop; ++ni) {
            for (uint32_t ki = 0; ki < kLoop; ++ki) {
                uint32_t bufIdx = ki & 1;
                uint32_t l1Event = (bufIdx == 0) ? L1_EVENT_0 : L1_EVENT_1;
                uint32_t l0Event = (bufIdx == 0) ? L0_EVENT_0 : L0_EVENT_1;

                auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf + bufIdx * baseM * baseK), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
                auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf + bufIdx * baseK * baseN), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
                auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf + bufIdx * baseM * baseK), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
                auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf + bufIdx * baseK * baseN), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));

                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(l1Event);
                Copy(copyGM2L1Atom, l1ATensor, gmASingle.Slice(MakeCoord(mi * baseM, ki * baseK), MakeShape(baseM, baseK)));
                Copy(copyGM2L1Atom, l1BTensor, gmBSingle.Slice(MakeCoord(ki * baseK, ni * baseN), MakeShape(baseK, baseN)));
                AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(l1Event);
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(l1Event);

                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(l0Event);
                Copy(copyL12L0AAtom, l0ATensor, l1ATensor);
                Copy(copyL12L0BAtom, l0BTensor, l1BTensor);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(l1Event);
                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(l0Event);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(l0Event);

                if (ki == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
                }
                MmadParams params{baseM, baseN, baseK, 0, (ki == 0)};
                Mmad(mmadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
                if (ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(l0Event);
            }

            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
            Copy(copyL0C2GMAtom, gmCSingle.Slice(MakeCoord(mi * baseM, ni * baseN), MakeShape(baseM, baseN)), l0CTensor);
            AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
        }
    }

    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_0);
    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_1);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_0);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_1);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
}

#endif


### 6.1 double buffer kernel 代码讲解

`TensorMmadDoubleBufferKernel` 在多核版本基础上，核心变化集中在缓冲分配和事件管理上：

#### 6.1.1 双倍缓冲

```cpp
__cbuf__ half l1ABuf[2 * baseM * baseK];   // 2份 L1A
__cbuf__ half l1BBuf[2 * baseK * baseN];   // 2份 L1B
__ca__ half l0ABuf[2 * baseM * baseK];     // 2份 L0A
__cb__ half l0BBuf[2 * baseK * baseN];     // 2份 L0B
__cc__ float l0CBuf[baseM * baseN];        // L0C 只需1份（输出不复用）
```

L1A/L1B 和 L0A/L0B 各分配两倍空间，L0C 仍为单份，因为同一个输出基本块的 K 方向累加结果始终写到同一块 L0C。

#### 6.1.2 交替缓冲选择

```cpp
uint32_t bufIdx = ki & 1;  // 偶数 K 分块用缓冲0，奇数用缓冲1
uint32_t l1Event = (bufIdx == 0) ? L1_EVENT_0 : L1_EVENT_1;
uint32_t l0Event = (bufIdx == 0) ? L0_EVENT_0 : L0_EVENT_1;

auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf + bufIdx * baseM * baseK), ...);
```

每个 K 分块根据 `ki & 1` 选择使用哪份缓冲。偶数分块用缓冲 0 和事件 0/2，奇数分块用缓冲 1 和事件 1/3。这样第 `ki` 个分块的搬运和第 `ki-1` 个分块的计算可以在不同缓冲上并行进行。

#### 6.1.3 独立事件 ID

```cpp
constexpr uint32_t L1_EVENT_0 = 0;
constexpr uint32_t L1_EVENT_1 = 1;
constexpr uint32_t L0_EVENT_0 = 2;
constexpr uint32_t L0_EVENT_1 = 3;
constexpr uint32_t L0C_EVENT_ID = 4;
```

两份缓冲各有独立的事件 ID，避免同步信号互相干扰。循环体内部的搬运和 Mmad 逻辑与多核版本一致，只是事件 ID 改为按 `bufIdx` 动态选择。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_double_buffer.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 */

#include "tensor_mmad_host.h"
#include "tensor_mmad_double_buffer_kernel.h"

namespace DoubleBufferConfig {
constexpr int32_t M = 8192;
constexpr int32_t N = 8192;
constexpr int32_t K = 8192;
constexpr int32_t singleM = 1024;
constexpr int32_t singleN = 2048;
constexpr int32_t singleK = 8192;
constexpr int32_t baseM = 128;
constexpr int32_t baseN = 256;
constexpr int32_t baseK = 64;
constexpr uint32_t NUM_BLOCKS = (M / singleM) * (N / singleN);
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace DoubleBufferConfig;

    constexpr size_t inputSize = static_cast<size_t>(M) * K * sizeof(half);
    constexpr size_t outputSize = static_cast<size_t>(M) * N * sizeof(half);

    auto launch = [](half *x, half *y, half *z, aclrtStream stream) {
        TensorMmadDoubleBufferKernel<M, N, K, singleM, singleN, singleK, baseM, baseN, baseK>
            <<<NUM_BLOCKS, nullptr, stream>>>(x, y, z);
    };
    return RunMatmulHost(inputSize, outputSize, "./output/double_buffer.bin", launch);
}


In [ ]:
!cd src/04_05_tensor_api_matmul && bash run.sh --npu-arch=dav-3510 --case=double_buffer

---

# 7. 性能对比

三个版本都已完成。可以在同一台 Ascend 950PR/DT 设备上分别运行单核、多核和 double buffer 版本，并使用 `msprof` 工具统计 kernel 整体和各流水阶段的耗时。

### 使用 msprof 采集性能数据

执行以下命令采集 kernel 性能数据（以单核版本为例）：

```bash
mkdir -p prof
msprof op --application="./build_out/tensor_mmad_single_core" --output=./prof/single_core
```

多核和 double buffer 版本替换 `--application` 参数为对应的可执行文件名即可。profiling 结果中的 `aic_mac_time` 统计 Cube 计算单元实际工作时间，`aic_mte1_time` 统计 L1→L0 搬运时间，`aic_mte2_time` 统计 GM→L1 搬运时间，均取所有 Cube Core 中的最大值。


对比时建议重点观察：单核到多核的加速是否明显；double buffer 相对多核版本是否继续改善；如果收益不明显，结合 L1/L0 占用、GM 带宽和同步开销分析原因。

| 优化方式 | Task Duration（us） | aic_mac_time（us） | aic_mte1_time（us） | aic_mte2_time（us） | aic_fixpipe_time（us） |
| --- | --- | --- | --- | --- | --- |
| 单核  | 220714.39 | 18394.5 | 6669.5 | 35158.7 | 208.1 |
| 多核 | 7558.52 | 3005.5  | 1089.7   | 6389.9   | 45.1 |
| 多核 + double buffer | 4622.9    | 3000.3    | 1150.9   | 4618.1   | 40.5 |

![性能对比](images/04_05_tensor_api_matmul/performance_comparison.png)


### 单核 → 多核

多核版本 Task Duration 从 `220714 us` 降到 `7559 us`，加速约 **29.2x**。Ascend 950PR/DT 有 32 个 Cube Core，本节多核切分正好生成 32 个输出 tile，接近理想并行加速。未达完整 32x 的原因：每个 Cube Core 仍需独立搬运 A/B 子块，GM 访问和 L1/L0 搬运存在开销，kernel 内部还有同步和尾部等待等非计算时间。

各流水单元分析：
- `aic_mac_time`：`18395 → 3006 us`，降幅约 6.1x，小于 Task Duration 的 29.2x。单核版本 Cube 计算单元大量空闲等待搬运，多核后每核计算更紧凑。
- `aic_mte2_time`：`35159 → 6390 us`，降幅约 5.5x，GM→L1 搬运开销随每核数据量减小而下降。
- `aic_mte1_time`：`6670 → 1090 us`，降幅约 6.1x，L1→L0 搬运量同样按核数缩小。
- `aic_fixpipe_time`：`208 → 45 us`，L0C→GM 搬出耗时极小，不是性能瓶颈。

### 多核 → double buffer

double buffer 版本 Task Duration 从 `7559 us` 进一步降到 `4623 us`，加速约 **1.64x**（耗时下降 38.8%）。多核并行用满 Cube Core 后，单核内部输入缓冲组织仍影响性能。双份 L1/L0 输入缓冲减少复用等待，搬运与计算重叠执行。

各流水单元分析：
- `aic_mac_time`：`3006 → 3000 us`，基本持平，计算量未变。
- `aic_mte2_time`：`6390 → 4618 us`，GM→L1 搬运等待明显减少，是端到端加速的主要来源（节省约 1772 us）。
- `aic_mte1_time`：`1090 → 1151 us`，略微升高，双缓冲下 L1→L0 需按 `bufIdx` 动态选择 L1 偏移，引入少量切换开销，远小于 GM→L1 收益。
- `aic_fixpipe_time`：`45 → 41 us`，L0C 搬出基本不变，不是瓶颈。

### 总结

单核→多核解决「多少 Cube Core 同时工作」问题，耗时下降一个数量级。多核→double buffer 解决「单核内部如何组织输入缓冲和流水等待」问题，在 32 核并行基础上继续压缩耗时。最终 double buffer 相对单核加速约 **47.7x**。

> **说明**：上述数据为 Ascend 950PR/DT 上 msprof 采集的参考值，具体数值随设备波动。`Task Duration` 为 kernel 端到端耗时，用于跨版本加速比对比；各 `aic_*_time` 为对应流水单元活跃时长（取所有核最大值），用于定位各阶段瓶颈。
>
> **关于 `aic_mac_time` 比值的补充说明**：单核到多核的 `aic_mac_time` 降幅约 6.1x（`18395 → 3006 us`），而非理想的 32x。这是因为单核版本中 Cube 计算单元在等待数据搬运时处于空闲状态，`aic_mac_time` 统计的是流水单元的活跃时长而非纯计算时间。多核版本中每核数据量减小、搬运更紧凑，Cube 计算单元的空闲等待减少，因此 `aic_mac_time` 下降。如需精确对比纯计算量，可参考 `M × N × K × 2 / (Cube 核峰值算力)` 的理论值。实际部署时请以目标设备上的 profiling 数据为准。


---

# 8. 课后练习与章节实践

## 课后练习

1. 说明多核版本中 `singleM=1024`、`singleN=2048` 对应的核心任务数，以及该任务数与 Ascend 950PR/DT 32 个 Cube Core 的关系。
2. 结合实践配置 `baseM=128`、`baseN=256`、`baseK=64`、`stepK=4`，计算 `mLoop/nLoop/kLoop` 和大包搬运次数，并分析该配置下 L1 缓冲区的占用大小。

## 章节实践

本实践同时作为**第 4 章综合实践**，用于检验本章对 Tensor API 数据流、模板化切分参数、Copy/Mmad 调用和 L1 缓冲管理的掌握情况。完成本实践后，建议结合 04.02（静态 Tensor CV 融合）和 04.04（CV 融合算子教程）的内容，综合运用本章所学的优化方法。

**任务：实现 stepK 大包搬运 kernel**

前面三个版本每次 K 循环都从 GM 搬入一个 `baseM × baseK` 的 A 块和一个 `baseK × baseN` 的 B 块，GM→L1 搬运指令数量等于 `kLoop`。大包搬运的思路是：一次从 GM 搬入 `stepK` 个基本块到 L1，后续 `stepK` 次 K 循环从 L1 中切取对应块搬入 L0，从而将 GM→L1 搬运指令数减少为 `kLoop / stepK`。

### 需修改文件

练习模板文件 `tensor_mmad_stepK_kernel.h` 保留三处 TODO，供你独立完成 stepK 大包搬运逻辑；可运行的参考实现放在 `tensor_mmad_stepK_answer_kernel.h`，`tensor_mmad_practice.asc` 默认引用参考实现，保证 `run.sh --case=practice` 可以直接完成编译、运行和校验。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">文件</th>
      <th align="left">修改内容</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>tensor_mmad_stepK_kernel.h</code></td>
      <td>补全三处 TODO：L1 缓冲区声明、GM→L1 大包搬运、L1→L0 切片搬运</td>
    </tr>
  </tbody>
</table>

### TODO 任务说明

1. **L1 缓冲区声明**：L1 缓冲区需要容纳 `stepK` 个基本块，大小为 `stepK * baseM * baseK`（A）和 `stepK * baseK * baseN`（B）。
2. **GM→L1 大包搬运**：每 `stepK` 次 K 循环才执行一次 `CopyGM2L1`，一次搬入 `stepK` 个基本块。提示：当 `ki % stepK == 0` 时执行搬运，GM 切片的 K 方向大小为 `stepK * baseK`。
3. **L1→L0 切片搬运**：每次 K 循环从 L1 大包中切出当前 `baseK` 块搬入 L0。提示：当前块在 L1 中的 K 方向偏移为 `(ki % stepK) * baseK`，使用 `Slice` 切取。

### 实践编写

在下方 code cell 中使用 `%%writefile` 写入练习模板。你可以将 TODO 补全后，让 `tensor_mmad_practice.asc` 改为引用该模板并运行校验；课程默认提供的 `practice` target 使用参考实现，便于直接验证 stepK 优化结果。


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_stepK_kernel.h
#ifndef TENSOR_MMAD_STEPK_KERNEL_H
#define TENSOR_MMAD_STEPK_KERNEL_H

#include "basic_api/kernel_operator_block_sync_intf.h"
#include "tensor_api/tensor.h"

template <
    int32_t M, int32_t N, int32_t K,
    int32_t singleM, int32_t singleN, int32_t singleK,
    int32_t baseM, int32_t baseN, int32_t baseK,
    int32_t stepK>
__cube__ __global__ void TensorMmadStepKKernel(__gm__ half *x, __gm__ half *y, __gm__ half *z)
{
    using namespace AscendC::Te;
    static_assert(M % singleM == 0 && N % singleN == 0 && K % singleK == 0);
    static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);
    static_assert(singleK % (baseK * stepK) == 0);

    constexpr uint32_t mCoreLoop = M / singleM;
    constexpr uint32_t nCoreLoop = N / singleN;
    constexpr uint32_t mLoop = singleM / baseM;
    constexpr uint32_t nLoop = singleN / baseN;
    constexpr uint32_t kLoop = singleK / baseK;

    uint32_t blockIdx = AscendC::GetBlockIdx();
    uint32_t mCoreIdx = blockIdx % mCoreLoop;
    uint32_t nCoreIdx = blockIdx / mCoreLoop;

    auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
    auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
    auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));

    auto gmASingle = gmATensor.Slice(MakeCoord(mCoreIdx * singleM, 0), MakeShape(singleM, singleK));
    auto gmBSingle = gmBTensor.Slice(MakeCoord(0, nCoreIdx * singleN), MakeShape(singleK, singleN));
    auto gmCSingle = gmCTensor.Slice(MakeCoord(mCoreIdx * singleM, nCoreIdx * singleN), MakeShape(singleM, singleN));

    // TODO(1): 声明 L1 缓冲区，每个需容纳 stepK 个基本块
    // 提示：l1ABuf 大小为 stepK * baseM * baseK，l1BBuf 大小为 stepK * baseK * baseN
    __cbuf__ half l1ABuf[/* 在此填写 */];
    __cbuf__ half l1BBuf[/* 在此填写 */];
    __ca__ half l0ABuf[baseM * baseK];
    __cb__ half l0BBuf[baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
    auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
    auto copyL12L0BAtom = MakeCopy(CopyL12L0B{}, CopyL12L0BTraitDefault{});
    auto copyL0C2GMAtom = MakeCopy(CopyL0C2GM{}, CopyL0C2GMTraitDefault{});
    auto mmadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

    auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, stepK * baseK));
    auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf), MakeFrameLayout<ZNLayoutPtn, half>(stepK * baseK, baseN));
    auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

    constexpr uint32_t L1_EVENT_ID = 0;
    constexpr uint32_t L0_EVENT_ID = 1;
    constexpr uint32_t L0C_EVENT_ID = 2;
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);

    for (uint32_t mi = 0; mi < mLoop; ++mi) {
        for (uint32_t ni = 0; ni < nLoop; ++ni) {
            for (uint32_t ki = 0; ki < kLoop; ++ki) {
                // TODO(2): 大包搬运 —— 每 stepK 次 K 循环才从 GM 搬入一次，每次搬入 stepK 个基本块
                // 提示：当 ki % stepK == 0 时执行 Copy
                //   A: gmASingle.Slice(MakeCoord(mi * baseM, ki * baseK), MakeShape(baseM, stepK * baseK))
                //   B: gmBSingle.Slice(MakeCoord(ki * baseK, ni * baseN), MakeShape(stepK * baseK, baseN))
                if (ki % stepK == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);

                    AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                    AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                }

                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);

                // TODO(3): 从 L1 大包中切出当前 baseK 块搬入 L0A/L0B
                // 提示：当前块在 L1 中的 K 偏移为 (ki % stepK) * baseK
                //   A: l1ATensor.Slice(MakeCoord(0, (ki % stepK) * baseK), MakeShape(baseM, baseK))
                //   B: l1BTensor.Slice(MakeCoord((ki % stepK) * baseK, 0), MakeShape(baseK, baseN))

                if ((ki + 1) % stepK == 0 || ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);

                if (ki == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
                }
                MmadParams params{baseM, baseN, baseK, 0, (ki == 0)};
                Mmad(mmadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
                if (ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
            }

            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
            Copy(copyL0C2GMAtom, gmCSingle.Slice(MakeCoord(mi * baseM, ni * baseN), MakeShape(baseM, baseN)), l0CTensor);
            AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
        }
    }

    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
}

#endif


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_stepK_answer_kernel.h
#ifndef TENSOR_MMAD_STEPK_ANSWER_KERNEL_H
#define TENSOR_MMAD_STEPK_ANSWER_KERNEL_H

#include "basic_api/kernel_operator_block_sync_intf.h"
#include "tensor_api/tensor.h"

template <
    int32_t M, int32_t N, int32_t K,
    int32_t singleM, int32_t singleN, int32_t singleK,
    int32_t baseM, int32_t baseN, int32_t baseK,
    int32_t stepK>
__cube__ __global__ void TensorMmadStepKAnswerKernel(__gm__ half *x, __gm__ half *y, __gm__ half *z)
{
    using namespace AscendC::Te;
    static_assert(M % singleM == 0 && N % singleN == 0 && K % singleK == 0);
    static_assert(singleM % baseM == 0 && singleN % baseN == 0 && singleK % baseK == 0);
    static_assert(singleK % (baseK * stepK) == 0);

    constexpr uint32_t mCoreLoop = M / singleM;
    constexpr uint32_t nCoreLoop = N / singleN;
    constexpr uint32_t mLoop = singleM / baseM;
    constexpr uint32_t nLoop = singleN / baseN;
    constexpr uint32_t kLoop = singleK / baseK;

    uint32_t blockIdx = AscendC::GetBlockIdx();
    uint32_t mCoreIdx = blockIdx % mCoreLoop;
    uint32_t nCoreIdx = blockIdx / mCoreLoop;

    auto gmATensor = MakeTensor(MakeMemPtr(x), MakeFrameLayout<NDLayoutPtn>(M, K));
    auto gmBTensor = MakeTensor(MakeMemPtr(y), MakeFrameLayout<DNLayoutPtn>(K, N));
    auto gmCTensor = MakeTensor(MakeMemPtr(z), MakeFrameLayout<NDLayoutPtn>(M, N));

    auto gmASingle = gmATensor.Slice(MakeCoord(mCoreIdx * singleM, 0), MakeShape(singleM, singleK));
    auto gmBSingle = gmBTensor.Slice(MakeCoord(0, nCoreIdx * singleN), MakeShape(singleK, singleN));
    auto gmCSingle = gmCTensor.Slice(MakeCoord(mCoreIdx * singleM, nCoreIdx * singleN), MakeShape(singleM, singleN));

    __cbuf__ half l1ABuf[stepK * baseM * baseK];
    __cbuf__ half l1BBuf[stepK * baseK * baseN];
    __ca__ half l0ABuf[baseM * baseK];
    __cb__ half l0BBuf[baseK * baseN];
    __cc__ float l0CBuf[baseM * baseN];

    auto copyGM2L1Atom = MakeCopy(CopyGM2L1{}, CopyGM2L1TraitDefault{});
    auto copyL12L0AAtom = MakeCopy(CopyL12L0A{}, CopyL12L0ATraitDefault{});
    auto copyL12L0BAtom = MakeCopy(CopyL12L0B{}, CopyL12L0BTraitDefault{});
    auto copyL0C2GMAtom = MakeCopy(CopyL0C2GM{}, CopyL0C2GMTraitDefault{});
    auto mMadAtom = MakeMmad(MmadOperation{}, MmadTraitDefault{});

    auto l1ATensor = MakeTensor(MakeMemPtr(l1ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, stepK * baseK));
    auto l1BTensor = MakeTensor(MakeMemPtr(l1BBuf), MakeFrameLayout<ZNLayoutPtn, half>(stepK * baseK, baseN));
    auto l0ATensor = MakeTensor(MakeMemPtr(l0ABuf), MakeFrameLayout<NZLayoutPtn, half>(baseM, baseK));
    auto l0BTensor = MakeTensor(MakeMemPtr(l0BBuf), MakeFrameLayout<ZNLayoutPtn, half>(baseK, baseN));
    auto l0CTensor = MakeTensor(MakeMemPtr(l0CBuf), MakeFrameLayout<NZLayoutPtn>(baseM, baseN));

    constexpr uint32_t L1_EVENT_ID = 0;
    constexpr uint32_t L0_EVENT_ID = 1;
    constexpr uint32_t L0C_EVENT_ID = 2;
    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);

    for (uint32_t mi = 0; mi < mLoop; ++mi) {
        for (uint32_t ni = 0; ni < nLoop; ++ni) {
            for (uint32_t ki = 0; ki < kLoop; ++ki) {
                if (ki % stepK == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                    Copy(copyGM2L1Atom, l1ATensor,
                        gmASingle.Slice(MakeCoord(mi * baseM, ki * baseK), MakeShape(baseM, stepK * baseK)));
                    Copy(copyGM2L1Atom, l1BTensor,
                        gmBSingle.Slice(MakeCoord(ki * baseK, ni * baseN), MakeShape(stepK * baseK, baseN)));
                    AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                    AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(L1_EVENT_ID);
                }

                AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
                Copy(copyL12L0AAtom, l0ATensor,
                    l1ATensor.Slice(MakeCoord(0, (ki % stepK) * baseK), MakeShape(baseM, baseK)));
                Copy(copyL12L0BAtom, l0BTensor,
                    l1BTensor.Slice(MakeCoord((ki % stepK) * baseK, 0), MakeShape(baseK, baseN)));
                if ((ki + 1) % stepK == 0 || ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(L0_EVENT_ID);

                if (ki == 0) {
                    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
                }
                MmadParams params{baseM, baseN, baseK, 0, (ki == 0)};
                Mmad(mMadAtom.with(params), l0CTensor, l0ATensor, l0BTensor);
                if (ki + 1 == kLoop) {
                    AscendC::SetFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
                }
                AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
            }

            AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(L0C_EVENT_ID);
            Copy(copyL0C2GMAtom, gmCSingle.Slice(MakeCoord(mi * baseM, ni * baseN), MakeShape(baseM, baseN)), l0CTensor);
            AscendC::SetFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
        }
    }

    AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(L1_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(L0_EVENT_ID);
    AscendC::WaitFlag<AscendC::HardEvent::FIX_M>(L0C_EVENT_ID);
}

#endif


In [ ]:
%%writefile src/04_05_tensor_api_matmul/tensor_mmad_practice.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 */

#include "tensor_mmad_host.h"
#include "tensor_mmad_stepK_answer_kernel.h"

namespace PracticeConfig {
constexpr int32_t M = 8192;
constexpr int32_t N = 8192;
constexpr int32_t K = 8192;
constexpr int32_t singleM = 1024;
constexpr int32_t singleN = 2048;
constexpr int32_t singleK = 8192;
constexpr int32_t baseM = 128;
constexpr int32_t baseN = 256;
constexpr int32_t baseK = 64;
constexpr int32_t stepK = 4;
constexpr uint32_t NUM_BLOCKS = (M / singleM) * (N / singleN);
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace PracticeConfig;

    constexpr size_t inputSize = static_cast<size_t>(M) * K * sizeof(half);
    constexpr size_t outputSize = static_cast<size_t>(M) * N * sizeof(half);

    auto launch = [](half *x, half *y, half *z, aclrtStream stream) {
        TensorMmadStepKAnswerKernel<M, N, K, singleM, singleN, singleK, baseM, baseN, baseK, stepK>
            <<<NUM_BLOCKS, nullptr, stream>>>(x, y, z);
    };
    return RunMatmulHost(inputSize, outputSize, "./output/practice.bin", launch);
}


In [ ]:
!cd src/04_05_tensor_api_matmul && bash run.sh --npu-arch=dav-3510 --case=practice

### 参考答案

课后练习和章节实践的参考答案如下。


In [ ]:
!cat answer/04_05_tensor_api_matmul/answer.md